# 快速入门-基于CANN快速跑通第一个基于ACL接口的分类推理应用
本快速入门指南以**图片分类**简单场景为例，讲解如何通过**ACL的C语言接口**开发推理应用，同时介绍开发过程中的核心概念与标准化实现流程。

整体开发与运行流程如下：

1. **环境准备**：配置程序运行所需环境，确保体验代码可以正常执行

2. **模型分析**：梳理待推理模型的输入输出规格、数据类型等关键信息

3. **模型转换**：将训练完成的原生模型转换为昇腾NPU支持的*.om离线模型文件

4. **输入数据准备**：准备推理所需图片数据，为代码正确性验证做准备

5. **ACL推理应用开发**：基于ACL接口完成图片分类推理应用的代码开发

6. **编译运行**：对开发完成的代码进行编译构建并执行，查看实际推理效果

#

> **Tips**  
> 本快速入门要求环境具备安装opencv，参考安装命令如下：  
> sudo apt-get install libopencv-dev

#
---

## 1. 环境准备
正式开始快速入门之前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，同时完成了代码目录的创建。保证能正常导入代码以及ATC等开发工具，完成模型转换以及代码的开发编译。

In [1]:
import os
!mkdir -p Sources
!bash -l -c 'source ~/.bashrc && env' > /tmp/shell_env.txt  # bash
with open("/tmp/shell_env.txt", "r") as f:
    for line in f.readlines():
        line = line.strip()
        if "=" in line and not line.startswith(("#", " ")):
            key, value = line.split("=", 1)
            os.environ[key] = value

---

## 2. 模型分析
图片分类应用通过AI模型推理获取图片所属类别并完成结果展示。本快速入门采用基于 PyTorch 框架训练的ResNet-50开源预训练模型，该模型基于ImageNet数据集训练，对应**ImageNet 1000类**分类标签，核心信息如下：

- 输入数据：RGB 格式、分辨率 224×224 的图片

- 输出数据：图片对应的类别标签及置信度

以下为模型结构片段（模型结构较大，仅展示头尾部分）：

<img src="./images/model_structure.png" alt="model_structure"  width="700px" >

可通过以下命令下载该 ResNet-50 开源预训练模型：

In [ ]:
# 华为云环境需要导入的环境变量
!export no_proxy=127.0.0.1,localhost,172.16.*,iam.cn-southwest-2.huaweicloud.com,pip.modelarts.private.com
!export NO_PROXY=127.0.0.1,localhost,172.16.*,iam.cn-southwest-2.huaweicloud.com,pip.modelarts.private.com

#下载resnet50.onnx模型文件
!wget https://obs-9be7.obs.cn-east-2.myhuaweicloud.com/003_Atc_Models/resnet50/resnet50.onnx -O ./Sources/resnet50.onnx

---

## 3. 模型转换
昇腾张量编译器（Ascend Tensor Compiler，ATC）是 CANN 异构计算架构下的核心模型转换工具，可将开源框架训练的网络模型、以及基于 Ascend IR 定义的单算子 JSON 描述文件，转换为昇腾 AI 处理器支持的.om 格式离线模型。

<img src="./images/atc_tool_architecture.png" alt="atc_tool_architecture"  width="700px" >

ATC 模型转换核心流程如下：

- 开源框架网络模型经 Parser 解析，转换为中间态 IR Graph；

- 中间态 IR 依次完成图准备、图拆分、图优化、图编译等处理，生成适配昇腾 AI 处理器的离线模型；

- 开发者可通过 ACL 接口加载生成的.om 模型文件，实现模型推理。

基于 ACL 接口开展推理应用开发时，需先将训练完成的开源框架模型转换为昇腾 AI 处理器可识别的.om 格式离线模型，具体转换命令如下：

In [ ]:
!atc --model=./Sources/resnet50.onnx --framework=5 --output=./Sources/resnet50 --input_shape="actual_input_1:1,3,224,224"  --soc_version=Ascend910B4

各参数说明如下：

- --model：ResNet-50 网络模型文件的存放路径

- --framework：原始模型框架类型，取值为 5 代表 ONNX 框架

- --output：resnet50.om 离线模型的生成路径（请留存该路径，后续应用开发需调用）

- --soc_version：适配的昇腾 AI 处理器版本，可通过对应命令查询芯片信息，在查询结果的 Name 字段值前添加 Ascend 前缀，即为该参数的配置值；示例：若查询到 Name 为 910B4，该参数填写为 Ascend910B4。

In [ ]:
!npu-smi info

---

## 4. 输入数据准备
本入门样例实现图片分类功能，验证时需准备一张ImageNet 1000 分类范围内的图片（也可自行替换），执行以下命令即可获取推理测试用图 dog1_1024_683.jpg。

In [ ]:
!export no_proxy=127.0.0.1,localhost,172.16.*,iam.cn-southwest-2.huaweicloud.com,pip.modelarts.private.combat
!export NO_PROXY=127.0.0.1,localhost,172.16.*,iam.cn-southwest-2.huaweicloud.com,pip.modelarts.private.com
!wget https://obs-9be7.obs.cn-east-2.myhuaweicloud.com/models/aclsample/dog1_1024_683.jpg -O ./Sources/dog1_1024_683.jpg

执行以下命令，即可查看测试图片的展示效果。

In [ ]:
%matplotlib inline

from IPython.display import Image
Image("./Sources/dog1_1024_683.jpg")

将 ResNet50 模型训练所用的**ImageNet 1000类**分类标签字典，实现并写入头文件 label.h

In [ ]:
%%writefile Sources/label.h

#ifndef modelInference_label_H
#define modelInference_label_H
const std::string label[] ={
"tenchTinca tinca", "goldfishCarassius auratus", "great white sharkwhite sharkman-eaterman-eating sharkCarcharodon carcharias", "tiger sharkGaleocerdo cuvieri", "hammerheadhammerhead shark", "electric raycrampfishnumbfishtorpedo", "stingray", "cock", "hen", "ostrichStruthio camelus", "bramblingFringilla montifringilla", "goldfinchCarduelis carduelis", "house finchlinnetCarpodacus mexicanus", "juncosnowbird", "indigo buntingindigo finchindigo birdPasserina cyanea", "robinAmerican robinTurdus migratorius", "bulbul", "jay", "magpie", "chickadee", "water ouzeldipper", "kite", "bald eagleAmerican eagleHaliaeetus leucocephalus", "vulture", "great grey owlgreat gray owlStrix nebulosa", "European fire salamanderSalamandra salamandra", "common newtTriturus vulgaris", "eft", "spotted salamanderAmbystoma maculatum", "axolotlmud puppyAmbystoma mexicanum", "bullfrogRana catesbeiana", "tree frogtree-frog", "tailed frogbell toadribbed toadtailed toadAscaphus trui", "loggerheadloggerhead turtleCaretta caretta", "leatherback turtleleatherbackleathery turtleDermochelys coriacea", "mud turtle", "terrapin", "box turtlebox tortoise", "banded gecko", "common iguanaiguanaIguana iguana", "American chameleonanoleAnolis carolinensis", "whiptailwhiptail lizard", "agama", "frilled lizardChlamydosaurus kingi", "alligator lizard", "Gila monsterHeloderma suspectum", "green lizardLacerta viridis", "African chameleonChamaeleo chamaeleon", "Komodo dragonKomodo lizarddragon lizardgiant lizardVaranus komodoensis", "African crocodileNile crocodileCrocodylus niloticus", "American alligatorAlligator mississipiensis", "triceratops", "thunder snakeworm snakeCarphophis amoenus", "ringneck snakering-necked snakering snake", "hognose snakepuff addersand viper", "green snakegrass snake", "king snakekingsnake", "garter snakegrass snake", "water snake", "vine snake", "night snakeHypsiglena torquata", "boa constrictorConstrictor constrictor", "rock pythonrock snakePython sebae", "Indian cobraNaja naja", "green mamba", "sea snake", "horned vipercerastessand viperhorned aspCerastes cornutus", "diamondbackdiamondback rattlesnakeCrotalus adamanteus", "sidewinderhorned rattlesnakeCrotalus cerastes", "trilobite", "harvestmandaddy longlegsPhalangium opilio", "scorpion", "black and gold garden spiderArgiope aurantia", "barn spiderAraneus cavaticus", "garden spiderAranea diademata", "black widowLatrodectus mactans", "tarantula", "wolf spiderhunting spider", "tick", "centipede", "black grouse", "ptarmigan", "ruffed grousepartridgeBonasa umbellus", "prairie chickenprairie grouseprairie fowl", "peacock", "quail", "partridge", "African greyAfrican grayPsittacus erithacus", "macaw", "sulphur-crested cockatooKakatoe galeritaCacatua galerita", "lorikeet", "coucal", "bee eater", "hornbill", "hummingbird", "jacamar", "toucan", "drake", "red-breasted merganserMergus serrator", "goose", "black swanCygnus atratus", "tusker", "echidnaspiny anteateranteater", "platypusduckbillduckbilled platypusduck-billed platypusOrnithorhynchus anatinus", "wallabybrush kangaroo", "koalakoala bearkangaroo bearnative bearPhascolarctos cinereus", "wombat", "jellyfish", "sea anemoneanemone", "brain coral", "flatwormplatyhelminth", "nematodenematode wormroundworm", "conch", "snail", "slug", "sea slugnudibranch", "chitoncoat-of-mail shellsea cradlepolyplacophore", "chambered nautiluspearly nautilusnautilus", "Dungeness crabCancer magister", "rock crabCancer irroratus", "fiddler crab", "king crabAlaska crabAlaskan king crabAlaska king crabParalithodes camtschatica", "American lobsterNorthern lobsterMaine lobsterHomarus americanus", "spiny lobsterlangousterock lobstercrawfishcrayfishsea crawfish", "crayfishcrawfishcrawdadcrawdaddy", "hermit crab", "isopod", "white storkCiconia ciconia", "black storkCiconia nigra", "spoonbill", "flamingo", "little blue heronEgretta caerulea", "American egretgreat white heronEgretta albus", "bittern", "crane", "limpkinAramus pictus", "European gallinulePorphyrio porphyrio", "American cootmarsh henmud henwater henFulica americana", "bustard", "ruddy turnstoneArenaria interpres", "red-backed sandpiperdunlinErolia alpina", "redshankTringa totanus", "dowitcher", "oystercatcheroyster catcher", "pelican", "king penguinAptenodytes patagonica", "albatrossmollymawk", "grey whalegray whaledevilfishEschrichtius gibbosusEschrichtius robustus", "killer whalekillerorcagrampussea wolfOrcinus orca", "dugongDugong dugon", "sea lion", "Chihuahua", "Japanese spaniel", "Maltese dogMaltese terrierMaltese", "PekinesePekingesePeke", "Shih-Tzu", "Blenheim spaniel", "papillon", "toy terrier", "Rhodesian ridgeback", "Afghan houndAfghan", "bassetbasset hound", "beagle", 
"bloodhound sleuthhound", "bluetick", "black-and-tan coonhound", "Walker houndWalker foxhound", "English foxhound", "redbone", "borzoiRussian wolfhound", "Irish wolfhound", "Italian greyhound", "whippet", "Ibizan houndIbizan Podenco", "Norwegian elkhoundelkhound", "otterhoundotter hound", "Salukigazelle hound", "Scottish deerhounddeerhound", "Weimaraner", "Staffordshire bullterrierStaffordshire bull terrier", "American Staffordshire terrierStaffordshire terrierAmerican pit bull terrierpit bull terrier", "Bedlington terrier", "Border terrier", "Kerry blue terrier", "Irish terrier", "Norfolk terrier", "Norwich terrier", "Yorkshire terrier", "wire-haired fox terrier", "Lakeland terrier", "Sealyham terrierSealyham", "AiredaleAiredale terrier", "cairncairn terrier", "Australian terrier", "Dandie DinmontDandie Dinmont terrier", "Boston bullBoston terrier", "miniature schnauzer", "giant schnauzer", "standard schnauzer", "Scotch terrierScottish terrierScottie", "Tibetan terrierchrysanthemum dog", "silky terrierSydney silky", "soft-coated wheaten terrier", "West Highland white terrier", "LhasaLhasa apso", "flat-coated retriever", "curly-coated retriever", "golden retriever", "Labrador retriever", "Chesapeake Bay retriever", "German short-haired pointer", "vizslaHungarian pointer", "English setter", "Irish setterred setter", "Gordon setter", "Brittany spaniel", "clumberclumber spaniel", "English springerEnglish springer spaniel", "Welsh springer spaniel", "cocker spanielEnglish cocker spanielcocker", "Sussex spaniel", "Irish water spaniel", "kuvasz", "schipperke", "groenendael", "malinois", "briard", "kelpie", "komondor", "Old English sheepdogbobtail", "Shetland sheepdogShetland sheep dogShetland", "collie", "Border collie", "Bouvier des FlandresBouviers des Flandres", "Rottweiler", "German shepherdGerman shepherd dogGerman police dogalsatian", "DobermanDoberman pinscher", "miniature pinscher", "Greater Swiss Mountain dog", "Bernese mountain dog", "Appenzeller", "EntleBucher", "boxer", "bull mastiff", "Tibetan mastiff", "French bulldog", "Great Dane", "Saint BernardSt Bernard", "Eskimo doghusky", "malamutemalemuteAlaskan malamute", "Siberian husky", "dalmatiancoach dogcarriage dog", "affenpinschermonkey pinschermonkey dog", "basenji", "pugpug-dog", "Leonberg", "NewfoundlandNewfoundland dog", "Great Pyrenees", "SamoyedSamoyede", "Pomeranian", "chowchow chow", "keeshond", "Brabancon griffon", "PembrokePembroke Welsh corgi", "CardiganCardigan Welsh corgi", "toy poodle", "miniature poodle", "standard poodle", "Mexican hairless", "timber wolfgrey wolfgray wolfCanis lupus", "white wolfArctic wolfCanis lupus tundrarum", "red wolfmaned wolfCanis rufusCanis niger", "coyoteprairie wolfbrush wolfCanis latrans", "dingowarrigalwarragalCanis dingo", "dholeCuon alpinus", "African hunting doghyena dogCape hunting dogLycaon pictus", "hyenahyaena", "red foxVulpes vulpes", "kit foxVulpes macrotis", "Arctic foxwhite foxAlopex lagopus", "grey foxgray foxUrocyon cinereoargenteus", "tabbytabby cat", "tiger cat", "Persian cat", "Siamese catSiamese", "Egyptian cat", "cougarpumacatamountmountain lionpainterpantherFelis concolor", "lynxcatamount", "leopardPanthera pardus", "snow leopardouncePanthera uncia", "jaguarpantherPanthera oncaFelis onca", "lionking of beastsPanthera leo", "tigerPanthera tigris", "cheetahchetahAcinonyx jubatus", "brown bearbruinUrsus arctos", "American black bearblack bearUrsus americanusEuarctos americanus", "ice bearpolar bearUrsus MaritimusThalarctos maritimus", "sloth bearMelursus ursinusUrsus ursinus", "mongoose", "meerkatmierkat", "tiger beetle", "ladybugladybeetlelady beetleladybirdladybird beetle", "ground beetlecarabid beetle", "long-horned beetlelongicornlongicorn beetle", "leaf beetlechrysomelid", "dung beetle", "rhinoceros beetle", "weevil", "fly", "bee", "antemmetpismire", "grasshopperhopper", "cricket", "walking stickwalkingstickstick insect", "cockroachroach", "mantismantid", "cicadacicala", "leafhopper", "lacewinglacewing fly", "dragonflydarning needledevil's darning needlesewing needlesnake feedersnake doctormosquito hawkskeeter hawk", "damselfly", "admiral", "ringletringlet butterfly", "monarchmonarch butterflymilkweed butterflyDanaus plexippus", "cabbage butterfly", "sulphur butterflysulfur butterfly", "lycaenidlycaenid butterfly", "starfishsea star", "sea urchin", "sea cucumberholothurian", "wood rabbitcottontailcottontail rabbit", "hare", "AngoraAngora rabbit", "hamster", "porcupinehedgehog", "fox squirreleastern fox squirrelSciurus niger", "marmot", "beaver", "guinea pigCavia cobaya", "sorrel", "zebra", "hogpiggruntersquealerSus scrofa", "wild boarboarSus scrofa",
"warthog", "hippopotamushipporiver horseHippopotamus amphibius", "ox", "water buffalowater oxAsiatic buffaloBubalus bubalis", "bison", "ramtup", "bighornbighorn sheepcimarronRocky Mountain bighornRocky Mountain sheepOvis canadensis", "ibexCapra ibex", "hartebeest", "impalaAepyceros melampus", "gazelle", "Arabian cameldromedaryCamelus dromedarius", "llama", "weasel", "mink", "polecatfitchfoulmartfoumartMustela putorius", "black-footed ferretferretMustela nigripes", "otter", "skunkpolecatwood pussy", "badger", "armadillo", "three-toed slothaiBradypus tridactylus", "orangutanorangorangutangPongo pygmaeus", "gorillaGorilla gorilla", "chimpanzeechimpPan troglodytes", "gibbonHylobates lar", "siamangHylobates syndactylusSymphalangus syndactylus", "guenonguenon monkey", "patashussar monkeyErythrocebus patas", "baboon", "macaque", "langur", "colobuscolobus monkey", "proboscis monkeyNasalis larvatus", "marmoset", "capuchinringtailCebus capucinus", "howler monkeyhowler", "titititi monkey", "spider monkeyAteles geoffroyi", "squirrel monkeySaimiri sciureus", "Madagascar catring-tailed lemurLemur catta", "indriindrisIndri indriIndri brevicaudatus", "Indian elephantElephas maximus", "African elephantLoxodonta africana", "lesser pandared pandapandabear catcat bearAilurus fulgens", "giant pandapandapanda bearcoon bearAiluropoda melanoleuca", "barracoutasnoek", "eel", "cohocohoecoho salmonblue jacksilver salmonOncorhynchus kisutch", "rock beautyHolocanthus tricolor", "anemone fish", "sturgeon", "gargarfishgarpikebillfishLepisosteus osseus", "lionfish", "pufferpufferfishblowfishglobefish", "abacus", "abaya", "academic gownacademic robejudge's robe", "accordionpiano accordionsqueeze box", "acoustic guitar", "aircraft carriercarrierflattopattack aircraft carrier", "airliner", "airshipdirigible", "altar", "ambulance", "amphibianamphibious vehicle", "analog clock", "apiarybee house", "apron", "ashcantrash cangarbage canwastebinash binash-binashbindustbintrash barreltrash bin", "assault rifleassault gun", "backpackback packknapsackpacksackrucksackhaversack", "bakerybakeshopbakehouse", "balance beambeam", "balloon", "ballpointballpoint penballpenBiro", "Band Aid", "banjo", "bannisterbanisterbalustradebalustershandrail", "barbell", "barber chair", "barbershop", "barn", "barometer", "barrelcask", "barrowgarden cartlawn cartwheelbarrow", "baseball", "basketball", "bassinet", "bassoon", "bathing capswimming cap", "bath towel", "bathtubbathing tubbathtub", "beach wagonstation wagonwagonestate carbeach waggonstation waggonwaggon", "beaconlighthousebeacon lightpharos", "beaker", "bearskinbusbyshako", "beer bottle", "beer glass", "bell cotebell cot", "bib", "bicycle-built-for-twotandem bicycletandem", "bikinitwo-piece", "binderring-binder", "binocularsfield glassesopera glasses", "birdhouse", "boathouse", "bobsledbobsleighbob", "bolo tiebolobola tiebola", "bonnetpoke bonnet", "bookcase", "bookshopbookstorebookstall", "bottlecap", "bow", "bow tiebow-tiebowtie", "brassmemorial tabletplaque", "brassierebrabandeau", "breakwatergroingroynemolebulwarkseawalljetty", "breastplateaegisegis", "broom", "bucketpail", "buckle", "bulletproof vest", "bullet trainbullet", "butcher shopmeat market", "cabhacktaxitaxicab", "caldroncauldron", "candletaperwax light", "cannon", "canoe", "can openertin opener", "cardigan", "car mirror", "carouselcarrouselmerry-go-roundroundaboutwhirligig", "carpenter's kittool kit", "carton", "car wheel", "cash machinecash dispenserautomated teller machineautomatic teller machineautomated tellerautomatic tellerATM", "cassette", "cassette player", "castle", "catamaran", "CD player", "cellovioloncello", "cellular telephonecellular phonecellphonecellmobile phone", "chain", "chainlink fence", "chain mailring mailmailchain armorchain armourring armorring armour", "chain sawchainsaw", "chest", "chiffoniercommode", "chimebellgong", "china cabinetchina closet", "Christmas stocking", "churchchurch building", "cinemamovie theatermovie theatremovie housepicture palace", "cleavermeat cleaverchopper", "cliff dwelling", "cloak", "cloggetapattensabot", "cocktail shaker", "coffee mug", "coffeepot", "coilspiralvolutewhorlhelix", "combination lock", "computer keyboardkeypad", "confectioneryconfectionarycandy store", "container shipcontainershipcontainer vessel", "convertible", "corkscrewbottle screw", "cornethorntrumpettrump", "cowboy boot", "cowboy hatten-gallon hat", "cradle", "crane", "crash helmet", "crate", "cribcot", "Crock Pot", "croquet ball", "crutch", "cuirass", "damdikedyke", "desk", "desktop computer", "dial telephonedial phone", "diapernappynapkin", "digital clock", "digital watch", "dining tableboard", 
"dishragdishcloth", "dishwasherdish washerdishwashing machine", "disk brakedisc brake", "dockdockagedocking facility", "dogsleddog sleddog sleigh", "dome", "doormatwelcome mat", "drilling platformoffshore rig", "drummembranophonetympan", "drumstick", "dumbbell", "Dutch oven", "electric fanblower", "electric guitar", "electric locomotive", "entertainment center", "envelope", "espresso maker", "face powder", "feather boaboa", "filefile cabinetfiling cabinet", "fireboat", "fire enginefire truck", "fire screenfireguard", "flagpoleflagstaff", "flutetransverse flute", "folding chair", "football helmet", "forklift", "fountain", "fountain pen", "four-poster", "freight car", "French hornhorn", "frying panfrypanskillet", "fur coat", "garbage truckdustcart", "gasmaskrespiratorgas helmet", "gas pumpgasoline pumppetrol pumpisland dispenser", "goblet", "go-kart", "golf ball", "golfcartgolf cart", "gondola", "gongtam-tam", "gown", "grand pianogrand", "greenhousenurseryglasshouse", "grilleradiator grille", "grocery storegroceryfood marketmarket", "guillotine", "hair slide", "hair spray", "half track", "hammer", "hamper", "hand blowerblow dryerblow drierhair dryerhair drier", "hand-held computerhand-held microcomputer", "handkerchiefhankiehankyhankey", "hard dischard diskfixed disk", "harmonicamouth organharpmouth harp", "harp", "harvesterreaper", "hatchet", "holster", "home theaterhome theatre", "honeycomb", "hookclaw", "hoopskirtcrinoline", "horizontal barhigh bar", "horse carthorse-cart", "hourglass", "iPod", "ironsmoothing iron", "jack-o'-lantern", "jeanblue jeandenim", "jeeplandrover", "jerseyT-shirttee shirt", "jigsaw puzzle", "jinrikisharicksharickshaw", "joystick", "kimono", "knee pad", "knot", "lab coatlaboratory coat", "ladle", "lampshadelamp shade", "laptoplaptop computer", "lawn mowermower", "lens caplens cover", "letter openerpaper knifepaperknife", "library", "lifeboat", "lighterlightigniterignitor", "limousinelimo", "linerocean liner", "lipsticklip rouge", "Loafer", "lotion", "loudspeakerspeakerspeaker unitloudspeaker systemspeaker system", "loupejeweler's loupe", "lumbermillsawmill", "magnetic compass", "mailbagpostbag", "mailboxletter box", "maillot", "maillottank suit", "manhole cover", "maraca", "marimbaxylophone", "mask", "matchstick", "maypole", "mazelabyrinth", "measuring cup", "medicine chestmedicine cabinet", "megalithmegalithic structure", "microphonemike", "microwavemicrowave oven", "military uniform", "milk can", "minibus", "miniskirtmini", "minivan", "missile", "mitten", "mixing bowl", "mobile homemanufactured home", "Model T", "modem", "monastery", "monitor", "moped", "mortar", "mortarboard", "mosque", "mosquito net", "motor scooterscooter", "mountain bikeall-terrain bikeoff-roader", "mountain tent", "mousecomputer mouse", "mousetrap", "moving van", "muzzle", "nail", "neck brace", "necklace", "nipple", "notebooknotebook computer", "obelisk", "oboehautboyhautbois", "ocarinasweet potato", "odometerhodometermileometermilometer", "oil filter", "organpipe organ", "oscilloscopescopecathode-ray oscilloscopeCRO", "overskirt", "oxcart", "oxygen mask", "packet", "paddleboat paddle", "paddlewheelpaddle wheel", "padlock", "paintbrush", "pajamapyjamapj'sjammies", "palace", "panpipepandean pipesyrinx", "paper towel", "parachutechute", "parallel barsbars", "park bench", "parking meter", "passenger carcoachcarriage", "patioterrace", "pay-phonepay-station", "pedestalplinthfootstall", "pencil boxpencil case", "pencil sharpener", "perfumeessence", "Petri dish", "photocopier", "pickplectrumplectron", "pickelhaube", "picket fencepaling", "pickuppickup truck", "pier", "piggy bankpenny bank", "pill bottle", "pillow", "ping-pong ball", "pinwheel", "piratepirate ship", "pitcherewer", "planecarpenter's planewoodworking plane", "planetarium", "plastic bag", "plate rack", "plowplough", "plungerplumber's helper", "Polaroid cameraPolaroid Land camera", "pole", "police vanpolice wagonpaddy wagonpatrol wagonwagonblack Maria", "poncho", "pool tablebilliard tablesnooker table", "pop bottlesoda bottle", "potflowerpot", "potter's wheel", "power drill", "prayer rugprayer mat", "printer", "prisonprison house", "projectilemissile", "projector", "puckhockey puck", "punching bagpunch bagpunching ballpunchball", "purse", "quillquill pen", "quiltcomfortercomfortpuff", "racerrace carracing car", "racketracquet", "radiator", "radiowireless", "radio telescoperadio reflector", "rain barrel", "recreational vehicleRVR.V.", "reel", "reflex camera", "refrigeratoricebox", "remote controlremote", "restauranteating houseeating placeeatery", "revolversix-gunsix-shooter", "rifle", "rocking chairrocker", "rotisserie", 
"rubber eraserrubberpencil eraser", "rugby ball", "ruleruler", "running shoe", "safe", "safety pin", "saltshakersalt shaker", "sandal", "sarong", "saxsaxophone", "scabbard", "scaleweighing machine", "school bus", "schooner", "scoreboard", "screenCRT screen", "screw", "screwdriver", "seat beltseatbelt", "sewing machine", "shieldbuckler", "shoe shopshoe-shopshoe store", "shoji", "shopping basket", "shopping cart", "shovel", "shower cap", "shower curtain", "ski", "ski mask", "sleeping bag", "slide ruleslipstick", "sliding door", "slotone-armed bandit", "snorkel", "snowmobile", "snowplowsnowplough", "soap dispenser", "soccer ball", "sock", "solar dishsolar collectorsolar furnace", "sombrero", "soup bowl", "space bar", "space heater", "space shuttle", "spatula", "speedboat", "spider webspider's web", "spindle", "sports carsport car", "spotlightspot", "stage", "steam locomotive", "steel arch bridge", "steel drum", "stethoscope", "stole", "stone wall", "stopwatchstop watch", "stove", "strainer", "streetcartramtramcartrolleytrolley car", "stretcher", "studio couchday bed", "stupatope", "submarinepigboatsubU-boat", "suitsuit of clothes", "sundial", "sunglass", "sunglassesdark glassesshades", "sunscreensunblocksun blocker", "suspension bridge", "swabswobmop", "sweatshirt", "swimming trunksbathing trunks", "swing", "switchelectric switchelectrical switch", "syringe", "table lamp", "tankarmy tankarmored combat vehiclearmoured combat vehicle", "tape player", "teapot", "teddyteddy bear", "televisiontelevision system", "tennis ball", "thatchthatched roof", "theater curtaintheatre curtain", "thimble", "thresherthrasherthreshing machine", "throne", "tile roof", "toaster", "tobacco shoptobacconist shoptobacconist", "toilet seat", "torch", "totem pole", "tow trucktow carwrecker", "toyshop", "tractor", "trailer trucktractor trailertrucking rigrigarticulated lorrysemi", "tray", "trench coat", "tricycletrikevelocipede", "trimaran", "tripod", "triumphal arch", "trolleybustrolley coachtrackless trolley", "trombone", "tubvat", "turnstile", "typewriter keyboard", "umbrella", "unicyclemonocycle", "uprightupright piano", "vacuumvacuum cleaner", "vase", "vault", "velvet", "vending machine", "vestment", "viaduct", "violinfiddle", "volleyball", "waffle iron", "wall clock", "walletbillfoldnotecasepocketbook", "wardrobeclosetpress", "warplanemilitary plane", "washbasinhandbasinwashbowllavabowash-hand basin", "washerautomatic washerwashing machine", "water bottle", "water jug", "water tower", "whiskey jug", "whistle", "wig", "window screen", "window shade", "Windsor tie", "wine bottle", "wing", "wok", "wooden spoon", "woolwoollenwoollen", "worm fencesnake fencesnake-rail fenceVirginia fence", "wreck", "yawl", "yurt", "web sitewebsiteinternet sitesite", "comic book", "crossword puzzlecrossword", "street sign", "traffic lighttraffic signalstoplight", "book jacketdust coverdust jacketdust wrapper", "menu", "plate", "guacamole", "consomme", "hot pothotpot", "trifle", "ice creamicecream", "ice lollylollylollipoppopsicle", "French loaf", "bagelbeigel", "pretzel", "cheeseburger", "hotdoghot dogred hot", "mashed potato", "head cabbage", "broccoli", "cauliflower", "zucchinicourgette", "spaghetti squash", "acorn squash", "butternut squash", "cucumbercuke", "artichokeglobe artichoke", "bell pepper", "cardoon", "mushroom", "Granny Smith", "strawberry", "orange", "lemon", "fig", "pineappleananas", "banana", "jackfruitjakjack", "custard apple", "pomegranate", "hay", "carbonara", "chocolate saucechocolate syrup", "dough", "meat loafmeatloaf", "pizzapizza pie", "potpie", "burrito", "red wine", "espresso", "cup", "eggnog", "alp", "bubble", "cliffdropdrop-off", "coral reef", "geyser", "lakesidelakeshore", "promontoryheadlandheadforeland", "sandbarsand bar", "seashorecoastseacoastsea-coast", "valleyvale", "volcano", "ballplayerbaseball player", "groombridegroom", "scuba diver", "rapeseed", "daisy", "yellow lady's slipperyellow lady-slipperCypripedium calceolusCypripedium parviflorum", "corn", "acorn", "hiprose hiprosehip", "buckeyehorse chestnutconker", "coral fungus", "agaric", "gyromitra", "stinkhorncarrion fungus", "earthstar", "hen-of-the-woodshen of the woodsPolyporus frondosusGrifola frondosa", "bolete", "earspikecapitulum", "toilet tissuetoilet paperbathroom tissue"
};
#endif // modelInference_label_H

---

## 5. ACL推理应用开发
ACL 推理应用的开发流程如下：

<img src="./images/acl_application_development_steps.png" alt="acl_application_development_steps" width="350px" >

模型准备工作已在前序环节完成，接下来正式开展 ACL 推理应用的开发流程。

### 5.1 包含头文件及全局定义

源文件的开头需要引入头文件并进行一些全局定义，包括：

- 依赖头文件，包括ACL的头文件、C或C++标准库的头文件、opencv的头文件。

- 引入命名空间，包括标准命名空间和opencv的命名空间。

具体代码如下所示：

In [ ]:
%%writefile Sources/main.cpp

// 头文件引入，其中acl是相对路径，要和后面编译文件中的头文件路径结合
#include <opencv2/opencv.hpp>
#include <dirent.h>
#include "acl/acl.h"
#include "label.h"
// 命名空间引入
using namespace cv;
using namespace std;
namespace {
    const float min_chn_0 = 123.675;
    const float min_chn_1 = 116.28;
    const float min_chn_2 = 103.53;
    const float var_reci_chn_0 = 0.0171247538316637;
    const float var_reci_chn_1 = 0.0175070028011204;
    const float var_reci_chn_2 = 0.0174291938997821;
}

### 5.2 资源初始化
资源初始化过程中，需要初始化如下数据：

- ACL初始化。

- 运行管理资源申请。

- 模型加载。

- 模型输入输出内存申请。

具体代码如下所示：

In [ ]:
%%writefile -a Sources/main.cpp

int main()
{
    // ACL初始化：
    const char *aclConfigPath = "";
    aclError ret = aclInit(aclConfigPath);
    
    // 运行管理资源申请:
    int32_t deviceId = 0;
    aclrtContext context;
    aclrtStream stream;
    ret = aclrtSetDevice(deviceId);
    ret = aclrtCreateContext(&context, deviceId);
    ret = aclrtCreateStream(&stream);
    aclrtRunMode runMode;
    ret = aclrtGetRunMode(&runMode);
    
    // 模型加载:
    const char* modelPath = "./resnet50.om";
    uint32_t modelId;
    ret = aclmdlLoadFromFile(modelPath, &modelId);
    aclmdlDesc *modelDesc = aclmdlCreateDesc();
    ret = aclmdlGetDesc(modelDesc, modelId);
    
    // 模型输入输出内存申请:
    aclmdlDataset *inputDataset = aclmdlCreateDataset();
    size_t inputIndex = 0;
    void* inputBuffer = nullptr;
    size_t inputBufferSize = aclmdlGetInputSizeByIndex(modelDesc, inputIndex);
    aclrtMalloc(&inputBuffer, inputBufferSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclDataBuffer *inputData = aclCreateDataBuffer(inputBuffer, inputBufferSize);
    ret = aclmdlAddDatasetBuffer(inputDataset, inputData);
  
    aclmdlDataset *outputDataset = aclmdlCreateDataset();
    size_t outputIndex = 0;
    void *outputBuffer = nullptr;
    size_t modelOutputSize = aclmdlGetOutputSizeByIndex(modelDesc, outputIndex);
    aclrtMalloc(&outputBuffer, modelOutputSize, ACL_MEM_MALLOC_HUGE_FIRST);
    aclDataBuffer *outputData = aclCreateDataBuffer(outputBuffer, modelOutputSize);
    ret = aclmdlAddDatasetBuffer(outputDataset, outputData);

### 5.3 输入数据处理
资源初始化完毕后，就需要读入输入数据并进行处理，本样例主要进行如下处理：

- 使用opencv读取图片

- 使用opencv将图片缩放到模型要求输入大小

- 对输入数据进行归一化、数据格式转换、色域转换操作，将BGR格式的读入图片转换成模型需要的RGB格式：

具体代码如下所示：

In [ ]:
%%writefile -a Sources/main.cpp

    // 使用opencv读取图片：
    const string imagePath = "./dog1_1024_683.jpg";
    Mat srcImage = imread(imagePath);
    
    // 使用opencv将图片缩放到模型要求输入大小：
    int32_t modelWidth = 224;
    int32_t modelHeight = 224;
    Mat resizedImage;
    resize(srcImage, resizedImage, Size(modelWidth, modelHeight));
    
    // 对输入数据进行归一化、数据格式转换、色域转换操作，将BGR格式的读入图片转换成模型需要的RGB格式：
    int32_t channel = resizedImage.channels();
    int32_t resizeHeight = resizedImage.rows;
    int32_t resizeWeight = resizedImage.cols;
    float meanRgb[3] = {min_chn_2, min_chn_1, min_chn_0};
    float stdRgb[3]  = {var_reci_chn_2, var_reci_chn_1, var_reci_chn_0};
    // create malloc of image, which is shape with NCHW
    float* imageBytes = (float*)malloc(channel * resizeHeight * resizeWeight * sizeof(float));
    memset(imageBytes, 0, channel * resizeHeight * resizeWeight * sizeof(float));
    uint8_t bgrToRgb=2;
    // image to bytes with shape HWC to CHW, and switch channel BGR to RGB
    for (int c = 0; c < channel; ++c)
    {
        for (int h = 0; h < resizeHeight; ++h)
        {
            for (int w = 0; w < resizeWeight; ++w)
            {
                int dstIdx = (bgrToRgb - c) * resizeHeight * resizeWeight + h * resizeWeight + w;
                imageBytes[dstIdx] =  static_cast<float>((resizedImage.at<cv::Vec3b>(h, w)[c] -
                                                         1.0f*meanRgb[c]) * 1.0f*stdRgb[c] );
            }
        }
    }

### 5.4 模型推理
输入数据处理完成后，就可以进行模型推理了，模型推理包括如下步骤：

- 将输入数据拷贝到device侧

- 执行模型推理

具体代码如下所示：

In [ ]:
%%writefile -a Sources/main.cpp

    // 将输入数据拷贝到device侧
    aclrtMemcpyKind kind;
    if (runMode == ACL_DEVICE)
    {
        kind = ACL_MEMCPY_DEVICE_TO_HOST;
    }else{
        kind = ACL_MEMCPY_HOST_TO_DEVICE;
    }
    ret = aclrtMemcpy(inputBuffer, inputBufferSize, imageBytes, inputBufferSize, kind);
    // 模型推理
    ret = aclmdlExecute(modelId, inputDataset, outputDataset);


### 5.5 输出数据处理
模型推理结束后，需要对推理后输出的数据进行分析，主要包含如下步骤：

- 获取所有输出的数据内容

- 将数据拷贝到Host侧

- 打印置信度

具体代码如下所示：

In [ ]:
%%writefile -a Sources/main.cpp

    // 获取所有输出的数据内容
    void *outHostData = nullptr;
    float *outData = nullptr;
    // size_t outputIndex = 0;
    aclDataBuffer* dataBuffer = aclmdlGetDatasetBuffer(outputDataset, outputIndex);
    void* data = aclGetDataBufferAddr(dataBuffer);
    uint32_t len = aclGetDataBufferSizeV2(dataBuffer);

    // 将数据拷贝到Host侧
    if (runMode == ACL_DEVICE)
    {
        kind = ACL_MEMCPY_DEVICE_TO_HOST;
    }else{
        kind = ACL_MEMCPY_HOST_TO_DEVICE;
    }
    aclrtMallocHost(&outHostData, len);
    ret = aclrtMemcpy(outHostData, len, data, len, kind);
    outData = reinterpret_cast<float*>(outHostData);

    // 打印置信度
    map<float, unsigned int, greater<float> > resultMap;
    for (unsigned int j = 0; j < len / sizeof(float); ++j) {
        resultMap[*outData] = j;
        outData++;
    }

    double totalValue=0.0;
    for (auto it = resultMap.begin(); it != resultMap.end(); ++it) {
        totalValue += exp(it->first);
    }
    float confidence = resultMap.begin()->first;
    unsigned int index = resultMap.begin()->second;
    string line = format("label:%d  conf:%lf  class:%s", index,
                 exp(confidence) / totalValue, label[index].c_str());
    cv::putText(srcImage, line, Point(0,35), cv::FONT_HERSHEY_TRIPLEX, 1.0, Scalar(255,0,0),2);
    string outputName = "out_" + imagePath;
    imwrite(outputName, srcImage);
    cout << outputName << endl;
    cout << line << endl;
    ret = aclrtFreeHost(outHostData);
    outHostData = nullptr;
    outData = nullptr;

### 5.6 资源释放
推理结束后，需要将所有资源进行释放，包括以下步骤：

- 模型输入数据释放

- 模型输出数据释放

- 模型管理资源释放

- 运行管理资源释放

- ACL去初始化

具体代码如下所示：

In [ ]:
%%writefile -a Sources/main.cpp

    // 模型输入数据释放
    aclrtFree(inputBuffer);
    inputBuffer = nullptr;
    (void)aclmdlDestroyDataset(inputDataset);
    inputDataset = nullptr;
    // 模型输出数据释放
    aclrtFree(outputBuffer);
    outputBuffer = nullptr;
    (void)aclmdlDestroyDataset(outputDataset);
    outputDataset = nullptr;
    // 模型管理资源释放
    ret = aclmdlDestroyDesc(modelDesc);
    ret = aclmdlUnload(modelId);
    // 运行管理资源释放
    ret = aclrtDestroyStream(stream);
    stream = nullptr;
    ret = aclrtDestroyContext(context);
    context = nullptr;
    ret = aclrtResetDevice(deviceId);
    // ACL去初始化
    ret = aclFinalize();
    return 0;
}

---

## 6. 编译运行
完成所有代码的开发工作后，即可进行编译运行。首先执行以下代码编译可执行文件：

In [ ]:
!g++ -std=c++11 -fPIC -O2 -Wall \
-I${ASCEND_HOME_PATH}/runtime/include/ \
-L${ASCEND_HOME_PATH}/runtime/lib64/stub/ \
./Sources/main.cpp -o ./Sources/main \
-lascendcl -lstdc++ -lopencv_core -lopencv_imgproc -lopencv_imgcodecs -ldl -lrt

再执行以下代码，进行模型推理，得到推理结果：

In [ ]:
!cd Sources && ./main

执行成功后，在屏幕上的关键提示信息示例如下，提示信息中的label表示类别标识、conf表示该分类的最大置信度，class表示所属类别。这些 值可能会根据版本、环境有所不同，请以实际情况为准：

```
[INFO] The sample starts to run
out_dog1_1024_683.jpg
label:162  conf:0.902209  class:beagle
[INFO] The program runs successfully
```

也可以执行如下命令查看推理后的输出图片

In [ ]:
%matplotlib inline

from IPython.display import Image
Image("./Sources/out_dog1_1024_683.jpg")